
# カタカナから漢字・アルファベットを推測するモデルを学習する
オープンソース辞書及び，Wikipediaから作成した漢字とカタカナのペアデータを用いて，Transformerで，スクラッチで学習する。

入力の文字（カタカナ）は一文字ずつに分割，また，出力の文字（漢字・アルファベットなど）も一文字ずつに分割して，学習する。

データセットは，漢字からカタカナを推測するモデルのデータを，漢字とカナを逆にしたものを利用する。

## データ準備
データは，[training.ipynb](./training.ipynb)で作成したものを利用する。


In [1]:
import os
ver='2.0o'
os.environ["VER"]=ver
os.environ['DATA']='../dict/data/data.'+os.environ["VER"]
os.environ['DIR']='dataset/dataset.'+os.environ["VER"]
os.environ['LOG']='logs/logs_r.'+os.environ["VER"]
os.environ['MODEL']='model/model_r.'+os.environ["VER"]

In [2]:
!echo $VER

2.0o


In [3]:
#training

In [4]:
!python ./transformer_model.py \
  --emb_size 512 \
  --nhead 8 \
  --ffn_hid_dim 2048 \
  --batch_size 64 \
  --num_encoder_layers 8 \
  --num_decoder_layers 8 \
  --lr 0.00002 \
  --dropout 0.3 \
  --num_epochs 100 \
  --device cuda \
  --earlystop_patient 3 \
  --output_dir $MODEL \
  --tensorboard_logdir $LOG \
  --source_lang kana \
  --target_lang kanji \
  --train_file $DIR/train.jsonl \
  --valid_file $DIR/valid.jsonl


[00:00:00] Pre-processing sequences       ██████████████████ 0        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 1        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 2        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 3        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 4        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 5        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 6        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 7        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 8        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 9        /        0
[00:00:00] Pre-processing sequences       ██████████████████ 10       /        0
[00:00:00] Pre-processing sequences       ██████████████████ 11       /        0
[00:00:00] Pre-processing se


[00:00:00] Pre-processing sequences       ██████████████████ 152      /        0
[00:00:00] Pre-processing sequences       ██████████████████ 257      /        0
[00:00:00] Pre-processing sequences       ██████████████████ 10753    /        0


[00:00:00] Pre-processing sequences       ██████████████████ 24833    /        0


[00:00:00] Pre-processing sequences       ██████████████████ 38913    /        0


[00:00:00] Pre-processing sequences       ██████████████████ 52993    /        0


[00:00:00] Pre-processing sequences       ██████████████████ 67073    /        0

... (317 outputs left out)

Epoch: 49, Train loss: 0.592, Val loss: 0.641, Epoch time = 6787.115s


Epoch: 50, Train loss: 0.591, Val loss: 0.638, Epoch time = 6784.150s


Epoch: 51, Train loss: 0.589, Val loss: 0.636, Epoch time = 6790.230s


Epoch: 47, Train loss: 0.596, Val loss: 0.635, Epoch time = 6666.831s


In [5]:
%load_ext tensorboard

In [6]:
%tensorboard --logdir logs/logs_r.$VER

## モデルを検証する
作成したモデルを，検証用データを用いて検証し，正解率を算出する


## パラメタ

- test_file    
jsonl形式の検証用データファイル

- model_file    
transformer_model.pyで作成したモデルファイル。

- outfile    
検証結果を格納するファイル

- device    
cpu限定

- nbest    
検証用データの漢字姓名を入力データとした時に，モデルが出力するカタカナ姓名を確率の高い方からnbest個出力する。beam_width以下。searchパラメタがbothかbeamの時有効

- beam_width    
ビームサーチのビーム幅。searchパラメタがbothかbeamの時有効

- max_len    
出力するカタカナ姓名の最大長さ

- search    
greedy,beam,bothから選択    
  - greedy    
貪欲法による探索    
  - beam    
ビームサーチによる探索    
  - both    
貪欲砲，ビームサーチ両方の探索    

In [ ]:
# 検証用データを実行
!python ./generate.py \
    --test_file $DIR/test.jsonl \
    --model_file $MODEL/checkpoint_best.pt \
    --outfile $DIR/generate.txt \
    --source_lang kana \
    --target_lang kanji \
    --device cpu \
    --nbest 5 \
    --beam_width 5 \
    --max_len 100 \
    --search beam

/mnt/src2/kanjikana-private/train/./generate.py:131: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  last_checkpoint = torch.load(self.args.model_file, map_location=torch.devi

/mnt/src2/kanjikana-private/venv/lib64/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [ ]:
# 検証結果を評価
!python evaluate.py --infile $DIR/generate_r.txt

In [9]:
!date

2026年  7月 17日 金曜日 12:25:51 JST


# モデルをJavaのDJL用に変換する
Pytorchで学習したモデルをJavaで使えるようにモデルの変換を行う。


In [1]:
import os
ver='2.0o'
os.environ["VER"]=ver
os.environ['DATA']='../dict/data/data.'+os.environ["VER"]
os.environ['DIR']='dataset/dataset.'+os.environ["VER"]
os.environ['LOG']='logs/logs_r.'+os.environ["VER"]
os.environ['MODEL']='model/model_r.'+os.environ["VER"]

In [2]:
!python conv_scripted.py  \
    --model_file=$MODEL/checkpoint_best.pt \
    --model_script=$MODEL/script_r_$VER.pt \
    --encoder=$MODEL/encoder_r_$VER.pt \
    --decoder=$MODEL/decoder_r_$VER.pt \
    --positional_encoding=$MODEL/positional_encoding_r_$VER.pt \
    --generator=$MODEL/generator_r_$VER.pt \
    --src_tok_emb=$MODEL/src_tok_emb_r_$VER.pt \
    --tgt_tok_emb=$MODEL/tgt_tok_emb_r_$VER.pt \
    --vocab_src=$MODEL/vocab_src_r_$VER.txt \
    --vocab_tgt=$MODEL/vocab_tgt_r_$VER.txt \
    --params=$MODEL/params_r_$VER.json \
    --device=cpu


/mnt/src2/kanjikana-private/train/conv_scripted.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_checkpoint = torch.load(self.args.model_file, map_location=torch.de

In [3]:
!date

2026年  7月 16日 木曜日 11:52:57 JST
